# Pilot2

Calculation of the KA and ACC for the original 6-class schema as well as 3-class schema. 
As this is the first pilot that corrected the mistakes from the previous pilot, these calculations also serve to verify results in the Pilot_evaluation table, along with Pilot1 notebook.

In [1]:
import pandas as pd
import csv
from sklearn.metrics import cohen_kappa_score
import krippendorff

df = pd.read_csv("../Datasets/pilot2.csv", delimiter=",")
df = df.drop(columns="id")
df.head()

,text,timestamp_tamara,flagged_tamara,comment_tamara,tag_tamara,timestamp_anze,flagged_anze,comment_anze,tag_anze,timestamp_katja,flagged_katja,comment_katja,tag_katja,rev_tamara,rev_anze,rev_katja,final_tag
0,"Jaz temu delu v celoti nasprotujem, mislim, da...",2023-02-28 10:58:59,NaN,NaN,Negative,2023-02-28 10:59:08,NaN,NaN,Negative,2023-02-28 10:58:57,NaN,NaN,Negative,Firm opposition - negative.,the speaker's argument,Opposition - negative.,Negative
1,"Izvršilna oblast, vaša naloga je izvršilna, op...",2023-02-28 11:00:05,NaN,No direct sentiment by the speaker. The statem...,P_Neutral,2023-02-28 10:59:45,NaN,Secons part is kind of negativ,N_Neutral,2023-02-28 10:59:35,NaN,"Statement, but sounds as criticism.",N_Neutral,"Okay, since I think I overlooked that it repea...",Secons part is kind of negativ,"Neutral statemenet, but second part implies cr...",N_Neutral
2,"Za katero pa se borimo, je možno, da nekoč dob...",2023-02-28 11:01:17,NaN,"No direct opinio by the speaker, but I think i...",Positive,2023-02-28 11:00:48,NaN,No special filinga negativ or positive,N_Neutral,2023-02-28 11:00:17,NaN,"Statement, but speaker weakly expresses hope",P_Neutral,Could be neutral since the opinion of the spea...,"Speakr opinion is mostly negativ, but it's not...","As opinion is simply implied, I would stick wi...",P_Neutral
3,Predlagani zakon ne pomeni bistvenih vsebinski...,2023-02-28 11:02:33,NaN,"First part weak negative, second part positive.",M_Positive,2023-02-28 11:01:44,NaN,The second part tells that the law is positiv,P_Neutral,2023-02-28 11:01:07,NaN,"First part slightly negative, second part most...",M_Positive,Leans on the positive side (because of the sec...,"First part slightly negative, second part most...","I agree that it has two (mixed) sentiments, fi...",M_Positive
4,"Torej, čez jarek ni mogoče skočiti v dveh skokih.",2023-02-28 11:03:17,NaN,"Clear sarcasm, no actual opinion by speaker.",P_Neutral,2023-02-28 11:01:59,NaN,S,Negative,2023-02-28 11:01:46,1.0,NaN,N_Neutral,"Here I put positive, because without context I...","Speakr opinion is mostly negativ,with a sarcasem","Reading it again, I agree that there is clear ...",N_Neutral(S)


### Krippendorffs Alpha for the original 6-class schema

In [2]:
tag_cols = ['tag_tamara', 'tag_katja', 'tag_anze']
tags = df[tag_cols]
tags.head()

,tag_tamara,tag_katja,tag_anze
0,Negative,Negative,Negative
1,P_Neutral,N_Neutral,N_Neutral
2,Positive,P_Neutral,N_Neutral
3,M_Positive,M_Positive,P_Neutral
4,P_Neutral,N_Neutral,Negative


In [3]:
unique_labels = pd.unique(tags.values.ravel())

# Create a mapping of labels to consistent numeric codes
label_to_code = {label: idx for idx, label in enumerate(unique_labels)}

# Map the labels to numeric codes for all annotators
for col in tag_cols:
    df[col + '_code'] = df[col].map(label_to_code)


In [4]:
coded_cols = [col + '_code' for col in tag_cols]
data_array = df[coded_cols].to_numpy().T
alpha_value = krippendorff.alpha(reliability_data=data_array, level_of_measurement='nominal')

print(f"Krippendorff's alpha (nominal): {alpha_value:.3f}")

Krippendorff's alpha (nominal): 0.332


In [5]:
for col in df.columns:
    if col.startswith("final_"):
        df[col] = df[col].apply(lambda x: x.strip().replace("(S)", "") if isinstance(x, str) and "(S)" in x else x)
        

In [6]:
df['final_tag'].value_counts()

final_tag
Negative      23
N_Neutral     12
P_Neutral      7
Positive       5
M_Negative     2
M_Positive     1
Name: count, dtype: int64

### Krippendorffs Alpha for the 3-class schema (Positive, Negative, Neutral)

In [7]:
mapping_3class = {
    'Positive': 'Positive',
    'M_Positive': 'Positive',    
    'Negative': 'Negative',
    'M_Negative': 'Negative',
    'P_Neutral': 'Neutral',
    'N_Neutral': 'Neutral',
    }

for col in tag_cols:
    df[col + '_sent'] = df[col].map(mapping_3class)

In [8]:
columns = ['tag_tamara_sent', 'tag_katja_sent', 'tag_anze_sent']
sent = df[columns]
sent.head()

,tag_tamara_sent,tag_katja_sent,tag_anze_sent
0,Negative,Negative,Negative
1,Neutral,Neutral,Neutral
2,Positive,Neutral,Neutral
3,Positive,Positive,Neutral
4,Neutral,Neutral,Negative


In [9]:
sent_unique = pd.unique(sent.values.ravel())
sent_to_code = {label: idx for idx, label in enumerate(sent_unique)}

for col in columns:
    df[col + '_code'] = df[col].map(sent_to_code)


In [10]:
sent_cols = [col + '_code' for col in columns]

sent_array = df[sent_cols].to_numpy().T
alpha_sent = krippendorff.alpha(reliability_data=sent_array, level_of_measurement='nominal')

print(f"Krippendorff's alpha (nominal): {alpha_sent:.3f}")


Krippendorff's alpha (nominal): 0.390
